# Entender la solución: Expense Policy Validator

Notebook para recorrer el pipeline pieza por pieza y poder defender cada decisión. Ejecuta las celdas en orden con `Shift + Enter`; después del setup, cada sección corre sola (en orden dentro de la sección; la 9 usa lo cargado en la 8).

- **Offline:** no lee `.env`, no usa API key. El clasificador se reproduce desde el cache en disco (`.cache/classifier/`); si falta una entrada, falla con un mensaje claro en vez de llamar a la red.
- **Código real:** todo se importa de `validator`, nada se reimplementa.
- **Respuestas:** `data_eval/gold.jsonl` solo se lee en las secciones 8 y 9, marcadas.

## 0. Setup y vista general

```
 receipts_input.jsonl  (sin respuestas: loader rechaza labels/notes/line_labels)
        |
        v
 [1] Normaliser          código    NFKC, quita caracteres de control, colapsa espacios
        |
        v
 [2] Classifier          LLM       1 llamada por recibo -> categoría, confianza, inglés, destinatario
        |                          (cacheado en disco; fail-closed)
        v
 [3] Policy retriever    código    tabla de predicados: qué reglas podrían aplicar (recall > precision)
        |
        v
 [4] Policy engine       código    37 reglas deterministas -> Check por regla -> disposición (EXC-01)
        |
        v
 [5] Exception reporter  código    JSON por recibo, cita el texto de la política tal cual
        |
        v
 output/report.jsonl, report.md

 Respuestas (gold.jsonl): solo validator.gold y validator.evaluation.* las importan.
 tests/test_no_leak.py lo verifica sobre el código del pipeline.
```

Lo único no determinista es el paso 2. Todo lo que decide `pass / review / flag` es código que puedes leer y testear.

In [1]:
import copy
import json
from collections import Counter
from unittest import mock

import pandas as pd

from validator import (
    classifier,
    disposition,
    engine,
    loader,
    normaliser,
    reporter,
    retriever,
)
from validator.evaluation.evaluate import _NoNetwork

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

ROOT = classifier.ROOT
RECEIPTS = loader.load_receipts()
BY_ID = {r["receipt_id"]: r for r in RECEIPTS}
CLEAN = [normaliser.normalise_receipt(r) for r in RECEIPTS]

C = classifier.classify(CLEAN, client=_NoNetwork(), cache_dir=classifier.CACHE_DIR)
if C.meta["failures"]:
    raise RuntimeError(f"cache incompleto en {classifier.CACHE_DIR}: {sorted(C.meta['failures'])}. "
                       "Este notebook no llama a la API; regenera el cache con validator.run y una key.")
RESULTS = engine.run_engine(RECEIPTS, C.categories, gift_for=C.gift_for, uncertain=C.uncertain)
RID = "RCP-016"  # el recibo de ejemplo: APAC, cerveza, cargo de servicio, rupias


def checks_table(result):
    return pd.DataFrame([{"rule": c.rule_id, "status": c.status, "kind": c.kind, "severity": c.severity,
                          "lines": ", ".join(c.line_ids), "detail": c.detail} for c in result.checks])


def what_if(receipt, categories=None, uncertain=None):
    """Re-run the engine on the whole batch with `receipt` swapped in (group rules need the other receipts)."""
    batch = [receipt if r["receipt_id"] == receipt["receipt_id"] else r for r in RECEIPTS]
    return engine.run_engine(batch, {**C.categories, **(categories or {})}, gift_for=C.gift_for,
                             uncertain=C.uncertain if uncertain is None else uncertain)[receipt["receipt_id"]]


def fired(result):
    return [(c.rule_id, c.severity or c.kind) for c in result.checks if c.status == "fired"]


print(f"{len(RECEIPTS)} recibos | cache hits {C.meta['cache_hits']} | llamadas {C.meta['calls']} | gasto ${C.meta['cost_usd']:.4f}")

32 recibos | cache hits 32 | llamadas 0 | gasto $0.0000


## 1. Los datos

`loader.load_receipts()` lee `data_split/receipts_input.jsonl`, que es el dataset original **sin** respuestas (lo genera `scripts/split_data.py`). Además recorre cada registro y revienta si encuentra `labels`, `notes` o `line_labels` a cualquier profundidad: el pipeline no puede ver las respuestas ni por accidente.

Qué mirar: 32 recibos, 79 líneas; 28 se califican (4 de timing quedan fuera de alcance, el brief los marca opcionales). De `validator.gold` aquí solo se importa esa constante con los 4 ids; `gold.jsonl` no se abre.

In [2]:
r = BY_ID[RID]
pd.Series({k: v for k, v in r.items() if k != "line_items"})

receipt_id                                      RCP-016
employee_id                                      E-3104
employee_country                                     IN
entity                                             APAC
submitted_expense_type             Client Entertainment
merchant_name                           Taj Spice House
mcc                                                5812
mcc_description           Eating places and restaurants
transaction_datetime                2026-04-13T20:30:00
currency                                            INR
fx_rate_to_usd                                    0.012
total_amount                                     5445.0
total_amount_usd                                  65.34
attendee_count                                        3
trip_id                                       TRIP-0201
trip_nights                                           2
pre_approval_ref                           PA-2026-0398
itemized                                        

In [3]:
pd.DataFrame(r["line_items"])

,line_id,description_raw,language,quantity,unit_price,amount,amount_usd
0,RCP-016-L1,Thali dinner,en,3,1200.0,3600.0,43.20
1,RCP-016-L2,Kingfisher beer 650ml,en,3,450.0,1350.0,16.20
2,RCP-016-L3,Service charge,en,1,495.0,495.0,5.94


In [4]:
from validator.gold import OUT_OF_SCOPE

n_lines = sum(len(x["line_items"]) for x in RECEIPTS)
print(f"recibos {len(RECEIPTS)} | líneas {n_lines} | calificados {len(RECEIPTS) - len(OUT_OF_SCOPE)} | fuera de alcance {sorted(OUT_OF_SCOPE)}")
pd.Series(Counter(li["language"] for x in RECEIPTS for li in x["line_items"]), name="líneas por idioma")

recibos 32 | líneas 79 | calificados 28 | fuera de alcance ['RCP-020', 'RCP-021', 'RCP-029', 'RCP-030']


en    68
es     2
fr     3
ja     2
de     3
pt     1
Name: líneas por idioma, dtype: int64

In [5]:
bad = {"receipt_id": "X", "line_items": [{"line_id": "X-L1", "line_labels": {}}]}
try:
    loader._check(bad, "X")
except ValueError as exc:
    print("loader rechaza:", exc)

loader rechaza: answer key ['line_labels'] found in input at X.line_items[0]


## 2. Normaliser

Limpieza determinista del texto antes de que lo vea el LLM: Unicode NFKC (letras y dígitos de ancho completo pasan a normales), fuera caracteres de control y de formato (zero-width, tabs), espacios colapsados. Es idempotente.

Por qué así: el cache del clasificador se indexa por el texto; dos variantes invisibles del mismo texto no deben dar dos llamadas ni dos respuestas. **No traduce**: la traducción la devuelve el LLM (`description_en`) en la misma llamada. Y `description_raw` original nunca se toca en el reporte.

Qué mirar: en este dataset el normaliser no cambia ninguna línea (los datos vienen limpios), así que el antes/después se ve con un texto inventado. Las líneas en otro idioma pasan igual.

In [6]:
changed = [(li["line_id"], li["description_raw"]) for x in RECEIPTS for li in x["line_items"]
           if normaliser.normalise(li["description_raw"]) != li["description_raw"]]
print("líneas del dataset que el normaliser cambia:", changed or "ninguna")

odd = "  Ｃｈｉａｎｔｉ\u200b  Classico\t750ml \n ﬁne  "
print(repr(odd))
print(repr(normaliser.normalise(odd)))
assert normaliser.normalise(normaliser.normalise(odd)) == normaliser.normalise(odd)

líneas del dataset que el normaliser cambia: ninguna
'  Ｃｈｉａｎｔｉ\u200b  Classico\t750ml \n ﬁne  '
'Chianti Classico 750ml fine'


In [7]:
ja = BY_ID["RCP-019"]
pd.DataFrame([{"line_id": li["line_id"], "language": li["language"], "raw": li["description_raw"],
               "normalised": normaliser.normalise(li["description_raw"]), "description_en (LLM)": C.english[li["line_id"]]}
              for li in ja["line_items"]])

,line_id,language,raw,normalised,description_en (LLM)
0,RCP-019-L1,ja,iPad Air 11インチ 128GB,iPad Air 11インチ 128GB,iPad Air 11-inch 128GB
1,RCP-019-L2,ja,タブレットケース,タブレットケース,Tablet case


## 3. Classifier (LLM) — replay del cache

Una llamada por recibo (no por línea): el modelo ve el merchant, MCC y tipo de gasto como contexto y devuelve, por línea, categoría, confianza, inglés y destinatario si el texto lo dice. Salida con JSON schema estricto (pydantic, `extra="forbid"`), validada y con un reintento.

Decisiones a defender:
- **El prompt es solo la taxonomía** (tabla y guía de `category-taxonomy.md`), sin ejemplos sacados de los recibos.
- **Confianza = bandas, no probabilidades.** `low/medium/high` se reportan como 0.3/0.7/0.9: números ordinales, no calibrados. Solo `low` dispara EXC-03.
- **`restricted` no se le pregunta al modelo**: sale de la tabla de la taxonomía según la categoría.
- **Fail-closed:** error de API, salida inválida o guard de gasto -> todas las líneas `other` e inciertas -> el recibo cae en `review`, nunca en `pass`.

**Todo lo de esta sección es replay:** se lee de `.cache/classifier/`, 0 llamadas, $0.

In [8]:
print(classifier.SYSTEM)

You classify the line items of one expense receipt for a policy engine.

Categories (category: typical examples):
- alcohol: beer, wine, spirits, cocktails, corkage
- tobacco: cigarettes, cigars, vaping products
- personal_electronics: headphones, tablets, smart watches, cameras, trackers, and accessories made for them (cases, chargers)
- gift: items made to be given: flowers, gift baskets, greeting cards, branded merchandise
- gift_card: gift cards, prepaid cards, vouchers
- household_goods: cleaning supplies, paper towels, kitchenware, drinkware, bedding, home décor
- personal_care: toiletries, cosmetics, haircuts, spa and massage
- clothing: apparel, footwear, fashion accessories (belts, ties, bags)
- gambling: casino chips, betting stakes, lottery
- jewelry: non-electronic watches, bracelets, precious metal items
- food_beverage: meals, non-alcoholic drinks, snacks, catering
- computer_accessory: cables, adapters, docks, mice, keyboards, monitors
- office_supplies: stationery, note

In [9]:
pd.DataFrame({"restricted": classifier.RESTRICTED}).T

,alcohol,tobacco,personal_electronics,gift,gift_card,household_goods,personal_care,clothing,gambling,jewelry,...,computer_accessory,office_supplies,lodging,ground_transport,airfare,laundry_service,health_otc,software_subscription,service_fee,other
restricted,True,True,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False


In [10]:
print("bandas -> número reportado:", reporter.CONFIDENCE)
print("campos por línea:", list(classifier.Line.model_fields))
print("modelo:", classifier.DEFAULT_MODEL, "| prompt", classifier.PROMPT_VERSION, "| effort", classifier.EFFORT)

bandas -> número reportado: {'rule': 1.0, 'high': 0.9, 'medium': 0.7, 'low': 0.3}
campos por línea: ['line_id', 'description_en', 'category', 'other_kind', 'note', 'confidence', 'recipient_kind', 'recipient_name']
modelo: claude-haiku-5-5 | prompt v2 | effort low


La entrada del cache es un hash de `modelo + versión de prompt + system prompt + schema + effort + input del recibo`. Si cambias cualquier cosa del prompt, el hash cambia y el cache ya no sirve: no hay forma de reusar respuestas viejas con un prompt nuevo sin querer.

In [11]:
payload = json.dumps(classifier._input(BY_ID[RID]), ensure_ascii=False, sort_keys=True)
path = classifier.CACHE_DIR / (classifier.cache_key(classifier.DEFAULT_MODEL, payload) + ".json")
print("input al modelo:", payload, "", f"cache: {path.relative_to(ROOT)}", sep="\n")
print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=1))

input al modelo:
{"attendee_count": 3, "currency": "INR", "employee_country": "IN", "entity": "APAC", "itemized": true, "line_items": [{"amount": 3600.0, "amount_usd": 43.2, "description": "Thali dinner", "language": "en", "line_id": "RCP-016-L1", "quantity": 3}, {"amount": 1350.0, "amount_usd": 16.2, "description": "Kingfisher beer 650ml", "language": "en", "line_id": "RCP-016-L2", "quantity": 3}, {"amount": 495.0, "amount_usd": 5.94, "description": "Service charge", "language": "en", "line_id": "RCP-016-L3", "quantity": 1}], "mcc": "5812", "mcc_description": "Eating places and restaurants", "merchant_name": "Taj Spice House", "submitted_expense_type": "Client Entertainment"}

cache: .cache/classifier/e52856f00ce6957609ac008abee135f134b2feb1080c5d4205ed5f301fc0b8b9.json
{
 "model": "claude-haiku-5-5",
 "prompt_version": "v2",
 "output": {
  "lines": [
   {
    "line_id": "RCP-016-L1",
    "description_en": "Thali dinner",
    "category": "food_beverage",
    "other_kind": "none",
    

In [12]:
def classified(rid):
    return pd.DataFrame([{"line_id": li["line_id"], "raw": li["description_raw"], "category": C.categories[li["line_id"]],
                          "restricted": C.restricted[li["line_id"]], "band": C.meta["confidence"][li["line_id"]],
                          "confidence": reporter.CONFIDENCE[C.meta["confidence"][li["line_id"]]],
                          "recipient": C.gift_for.get(li["line_id"]), "note": C.notes.get(li["line_id"])}
                         for li in BY_ID[rid]["line_items"]])


classified(RID)

,line_id,raw,category,restricted,band,confidence,recipient,note
0,RCP-016-L1,Thali dinner,food_beverage,False,high,0.9,None,None
1,RCP-016-L2,Kingfisher beer 650ml,alcohol,True,high,0.9,None,None
2,RCP-016-L3,Service charge,service_fee,False,high,0.9,None,None


**Recibo no itemizado:** si el recibo no trae detalle, la categoría es `other` por regla (texto de EXC-03), no por el modelo. La banda queda en `rule` (1.0) y la línea nunca se marca incierta; DOC-01 es quien lo marca.

In [13]:
classified("RCP-023")

,line_id,raw,category,restricted,band,confidence,recipient,note
0,RCP-023-L1,Restaurant charge - no itemization provided,other,False,rule,1.0,None,non-itemized charge: contents unknown


In [14]:
print({k: C.meta[k] for k in ("provider", "model", "calls", "cache_hits", "input_tokens", "output_tokens", "cost_usd")})
print("bandas en el dataset:", Counter(C.meta["confidence"].values()))
print("guard de gasto por corrida: $", classifier.BUDGET_USD)

{'provider': 'anthropic', 'model': 'claude-haiku-5-5', 'calls': 0, 'cache_hits': 32, 'input_tokens': 0, 'output_tokens': 0, 'cost_usd': 0.0}
bandas en el dataset: Counter({'high': 78, 'rule': 1})
guard de gasto por corrida: $ 0.5


**Fail-closed y guard de gasto, sin red.** Cambio la descripción de una línea para forzar un cache miss y uso un directorio temporal como cache. Con `budget_usd=0` el guard corta antes de llamar (estima el peor caso: 1 token por carácter + `MAX_TOKENS` de salida). Con el cliente `_NoNetwork` (el mismo que usa `evaluate.py`, de ahí su mensaje) la "llamada" revienta sin tocar la red. En los dos casos el recibo queda todo `other` + incierto, y el engine lo manda a `review`.

In [15]:
import tempfile

miss = copy.deepcopy(BY_ID[RID])
miss["line_items"][1]["description_raw"] = "Kingfisher beer 650ml (edited)"
with tempfile.TemporaryDirectory() as tmp:
    guard = classifier.classify([miss], client=_NoNetwork(), cache_dir=tmp, budget_usd=0)
    boom = classifier.classify([miss], client=_NoNetwork(), cache_dir=tmp)
print("budget 0  ->", guard.meta["failures"], "| llamadas", guard.meta["calls"])
print("sin red   ->", boom.meta["failures"])
print("categorías:", guard.categories, "| inciertas:", guard.uncertain)
res = engine.run_engine([miss], guard.categories, uncertain=guard.uncertain)[RID]
print("disposición del engine:", res.disposition, fired(res))

budget 0  -> {'RCP-016': 'budget guard: $0.0000 spent of $0'} | llamadas 0
sin red   -> {'RCP-016': 'RuntimeError: cache miss: evaluate.py makes no model calls without --allow-calls'}
categorías: {'RCP-016-L2': 'other', 'RCP-016-L1': 'other', 'RCP-016-L3': 'other'} | inciertas: {'RCP-016': {'RCP-016-L2', 'RCP-016-L1', 'RCP-016-L3'}}
disposición del engine: review [('EXC-03', 'signal')]


## 4. Policy retriever

Una tabla de predicados en código, una por regla, leída de la frase de alcance de cada regla en `data/policy-docs`. Sin vector DB, sin modelo. Pregunta "¿el tema de la regla está presente?" (una categoría, un MCC, una entidad...), nunca "¿se rompe?": eso es del engine.

Por qué la recall se **afirma** y no se confía: el engine igual evalúa las 37 reglas (son baratas) y si una regla dispara sin estar entre los candidatos, lanza `RuntimeError`. Un predicado demasiado estrecho falla fuerte en vez de esconder una violación. Los candidatos sirven para el reporte (`rules_considered`) y como prueba de que el retriever entiende el recibo.

Qué mirar: cuántas reglas quedan de 37 y la razón de cada una. Sin contar las 4 TIME-* (opcionales), son ~7 de 33 en promedio.

In [16]:
cands = retriever.candidate_rules(BY_ID[RID], C.categories, C.gift_for)
pd.DataFrame(cands.items(), columns=["rule", "why"])

,rule,why
0,ALC-01,"alcohol line, or a bar or liquor store"
1,ALC-02,"alcohol line, or a bar or liquor store"
2,ALC-03,"alcohol line, or a bar or liquor store"
3,REG-01,"alcohol line, APAC employee"
4,DOC-02,"every receipt needs a merchant, date and total"
5,DOC-03,declared 'Client Entertainment' vs the line categories
6,DOC-04,currency INR is not USD
7,MEAL-01,declared meal type 'Client Entertainment'
8,MEAL-02,declared meal type 'Client Entertainment'
9,REG-02,"meal, APAC employee"


In [17]:
sizes = pd.Series({rid: len(res.considered) for rid, res in RESULTS.items()})
no_time = pd.Series({rid: sum(not k.startswith("TIME-") for k in res.considered) for rid, res in RESULTS.items()})
print(f"reglas totales {len(engine.EXPECTED)} ({sum(not r.startswith('TIME-') for r in engine.EXPECTED)} sin TIME-*)")
print(f"candidatas por recibo: media {sizes.mean():.2f}, min {sizes.min()}, max {sizes.max()} | sin TIME-*: media {no_time.mean():.2f}")

reglas totales 37 (33 sin TIME-*)
candidatas por recibo: media 9.75, min 6, max 16 | sin TIME-*: media 7.19


In [18]:
real = retriever.candidate_rules


def narrow(r, cats, gift=None):
    return {k: v for k, v in real(r, cats, gift).items() if k != "REG-01"}


with mock.patch.object(retriever, "candidate_rules", narrow):
    try:
        engine.run_engine(RECEIPTS, C.categories, gift_for=C.gift_for)
    except RuntimeError as exc:
        print("el engine detecta el hueco de recall:", exc)

el engine detecta el hueco de recall: RCP-016: retriever missed rule REG-01


## 5. Policy engine (determinista)

Cada familia de reglas devuelve un `Check` por regla y por recibo: `fired`, `not_fired`, `not_applicable` o `superseded`, con un `detail` que tiene los números de la decisión. La disposición sale de los `fired` (EXC-01): cualquier excepción medium/high/critical -> `flag`; solo low o solo señales -> `review`; nada -> `pass`.

Puntos a defender:
- **Cobertura probada (D-008):** `verdict([])` sería `pass`, así que el engine verifica que cada una de las 37 reglas corrió exactamente una vez por recibo, si no revienta.
- **Umbrales estrictos:** "más de $30" es `>`; exactamente el tope pasa. Montos con `Decimal`/`Fraction`, no float.
- **Supersesión:** REG-01 (APAC: alcohol nunca) reemplaza ALC-01/02/03; REG-02/REG-03 reemplazan MEAL-01. Superseded nunca cuenta.
- **EXC-03:** las líneas `low` se sacan de todas las demás reglas y el recibo recibe una señal de revisión.
- **Aislamiento de fallos:** si una familia revienta, solo sus reglas se pierden: aparece `ERR-<familia>` como señal (nunca `pass`) y lo demás sigue contando.

Qué mirar abajo: en RCP-016 dispara REG-01 (high) y ALC-01/02/03 quedan `superseded`; MEAL-01 también, por REG-02.

In [19]:
print(RID, "->", RESULTS[RID].disposition)
checks_table(RESULTS[RID])

RCP-016 -> flag


,rule,status,kind,severity,lines,detail
0,ALC-01,superseded,exception,NaN,,replaced by REG-01 for APAC employees
1,ALC-02,superseded,exception,NaN,,replaced by REG-01 for APAC employees
2,ALC-03,superseded,exception,NaN,,replaced by REG-01 for APAC employees
3,REG-01,fired,exception,high,RCP-016-L2,APAC employee: alcohol is never reimbursable
4,MCC-01,not_fired,exception,NaN,,MCC 5812 is not high risk
5,MCC-02,not_fired,signal,NaN,,MCC 5812 is not medium risk
6,TECH-01,not_applicable,exception,NaN,,no personal_electronics line
7,TECH-02,not_applicable,exception,NaN,,no computer_accessory line
8,TECH-03,not_applicable,exception,NaN,,no mobile_handset line
9,TECH-04,not_applicable,exception,NaN,,no software_subscription line


In [20]:
checks_table(RESULTS[RID])["status"].value_counts()

status
not_applicable    20
not_fired         12
superseded         4
fired              1
Name: count, dtype: int64

**Disposición (EXC-01)** es una función pura de severidades y número de señales:

In [21]:
cases = [([], 0), ([], 1), (["low"], 0), (["low", "medium"], 0), (["critical"], 2)]
pd.DataFrame([{"severities": s, "signals": n, "disposition": disposition.disposition(s, n)} for s, n in cases])

,severities,signals,disposition
0,[],0,pass
1,[],1,review
2,[low],0,review
3,"[low, medium]",0,flag
4,[critical],2,flag


**Umbral estricto.** RCP-015: Barolo $110 entre 2 asistentes = $55 > $30 -> ALC-02. Con $60 queda exactamente en $30 por persona y no dispara. Con $60.01 el `detail` muestra $30.00 (redondeo al imprimir) pero dispara: la comparación es exacta con `Decimal`, no sobre el texto.

In [22]:
for usd in (110.0, 60.01, 60.0):
    x = copy.deepcopy(BY_ID["RCP-015"])
    x["line_items"][1]["amount_usd"] = usd
    alc02 = next(c for c in what_if(x).checks if c.rule_id == "ALC-02")
    print(f"${usd:>6}: {alc02.status:<9} {alc02.detail}")

$ 110.0: fired     $110.0 / 2 = $55.00 per attendee vs $30
$ 60.01: fired     $60.01 / 2 = $30.00 per attendee vs $30
$  60.0: not_fired $60.0 / 2 = $30.00 per attendee vs $30


**EXC-03.** Si el clasificador hubiera dicho `low` para la cerveza, esa línea sale de todas las reglas: REG-01 ya no puede dispararse y el recibo pasa de `flag` a `review` con EXC-03 citando la línea. Un humano la mira, pero baja de severidad: vale la pena tenerlo claro en la defensa.

In [23]:
unsure = what_if(BY_ID[RID], uncertain={RID: {"RCP-016-L2"}})
print(unsure.disposition, fired(unsure))
checks_table(unsure).query("status == 'fired' or rule == 'REG-01'")

review [('EXC-03', 'signal')]


,rule,status,kind,severity,lines,detail
3,REG-01,not_applicable,exception,None,,no alcohol line
36,EXC-03,fired,signal,None,RCP-016-L2,category not confident: line left out of every other rule


**Aislamiento de fallos.** Pongo `entity="GLOBAL"` pero dejo el país en India. Revientan dos familias: regional (entidad y país no cuadran) y meals (no hay moneda local conocida para IN, y prefiere fallar a adivinar el tope). Cada una se pierde sola como `ERR-<familia>`, el recibo va a `review` y el resto de reglas se evalúa normal.

In [24]:
broken = copy.deepcopy(BY_ID[RID])
broken["entity"] = "GLOBAL"
r_err = what_if(broken)
print(r_err.disposition, "| error:", r_err.error)
print(fired(r_err))

review | error: regional: ValueError: entity 'GLOBAL' does not match employee_country 'IN' (expected APAC); meals: KeyError: 'IN'
[('ERR-regional', 'signal'), ('ERR-meals', 'signal')]


**Juega tú.** Cambia los valores y vuelve a ejecutar la celda. Ideas: `ENTITY, COUNTRY = "GLOBAL", "US"` (la misma cerveza pasa: cliente con 3 asistentes y $5.40 por persona); `BEER_CATEGORY = "food_beverage"`; `ATTENDEES = 1`; `BEER_USD = 200`.

In [25]:
ENTITY, COUNTRY = "APAC", "IN"
BEER_CATEGORY = "alcohol"
BEER_USD = 16.2
ATTENDEES = 3

x = copy.deepcopy(BY_ID[RID])
x.update(entity=ENTITY, employee_country=COUNTRY, attendee_count=ATTENDEES)
x["line_items"][1]["amount_usd"] = BEER_USD
out = what_if(x, categories={"RCP-016-L2": BEER_CATEGORY})
print(out.disposition, fired(out), "| error:", out.error)
checks_table(out).query("status != 'not_applicable'")

flag [('REG-01', 'high')] | error: None


,rule,status,kind,severity,lines,detail
0,ALC-01,superseded,exception,NaN,,replaced by REG-01 for APAC employees
1,ALC-02,superseded,exception,NaN,,replaced by REG-01 for APAC employees
2,ALC-03,superseded,exception,NaN,,replaced by REG-01 for APAC employees
3,REG-01,fired,exception,high,RCP-016-L2,APAC employee: alcohol is never reimbursable
4,MCC-01,not_fired,exception,NaN,,MCC 5812 is not high risk
5,MCC-02,not_fired,signal,NaN,,MCC 5812 is not medium risk
17,DOC-01,not_fired,exception,NaN,,"$65.34 vs $75, itemized=True"
18,DOC-02,not_fired,exception,NaN,,"merchant name, date and total are recorded"
19,DOC-03,not_fired,exception,NaN,,"'Client Entertainment' covers ['alcohol', 'food_beverage']; lines are ['alcohol', 'food_beverage']"
20,DOC-04,not_fired,exception,NaN,,INR at rate 0.012


## 6. Exception reporter

Convierte el `Result` del engine en el JSON por recibo del brief. Sin LLM. Cada excepción lleva regla, severidad, documento de política y el **texto de la regla citado tal cual** del `.md` (se parsea en runtime: el documento de una regla es el archivo donde está su párrafo `**ID — título.**`). La explicación es el `detail` del engine. Re-verifica EXC-04: las líneas citadas existen y nunca son `service_fee`.

Qué mirar: el registro coincide con `output/report.jsonl`, y cada cita aparece literal en su archivo de política.

In [26]:
rep = reporter.report(BY_ID[RID], RESULTS[RID], C)
committed = {json.loads(x)["receipt_id"]: json.loads(x) for x in (ROOT / "output" / "report.jsonl").read_text(encoding="utf-8").splitlines()}
assert {**rep, "provider": C.meta["provider"]} == committed[RID]
print(json.dumps(rep, ensure_ascii=False, indent=1))

{
 "receipt_id": "RCP-016",
 "disposition": "flag",
 "line_items": [
  {
   "line_id": "RCP-016-L1",
   "description_raw": "Thali dinner",
   "amount_usd": 43.2,
   "category": "food_beverage",
   "restricted": false,
   "confidence": 0.9,
   "confidence_band": "high",
   "description_en": "Thali dinner"
  },
  {
   "line_id": "RCP-016-L2",
   "description_raw": "Kingfisher beer 650ml",
   "amount_usd": 16.2,
   "category": "alcohol",
   "restricted": true,
   "confidence": 0.9,
   "confidence_band": "high",
   "description_en": "Kingfisher beer 650ml"
  },
  {
   "line_id": "RCP-016-L3",
   "description_raw": "Service charge",
   "amount_usd": 5.94,
   "category": "service_fee",
   "restricted": false,
   "confidence": 0.9,
   "confidence_band": "high",
   "description_en": "Service charge"
  }
 ],
 "exceptions": [
  {
   "rule_id": "REG-01",
   "policy_doc": "regional-addendum.md",
   "severity": "high",
   "line_ids": [
    "RCP-016-L2"
   ],
   "evidence": "Alcohol is **never** rei

In [27]:
for e in rep["exceptions"]:
    doc = (reporter.POLICY_DIR / e["policy_doc"]).read_text(encoding="utf-8")
    print(e["rule_id"], e["policy_doc"], "| cita literal en el archivo:", e["evidence"] in doc)
    print("  ", e["evidence"])

REG-01 regional-addendum.md | cita literal en el archivo: True
   Alcohol is **never** reimbursable for APAC-entity employees, including
within client entertainment. This overrides ALC-01 and ALC-02: where those rules would have
permitted alcohol with an attendee count, REG-01 still makes it a **high** exception. For APAC employees REG-01 replaces ALC-01, ALC-02 and
ALC-03: report REG-01 instead of those three. Rules about the merchant, such as MCC-01, still apply.


In [28]:
every = [e for x in committed.values() for e in x["exceptions"]]
ok = sum(e["evidence"] in (reporter.POLICY_DIR / e["policy_doc"]).read_text(encoding="utf-8") for e in every)
print(f"citas literales en todo el reporte: {ok}/{len(every)}")

citas literales en todo el reporte: 28/28


## 7. End to end, los 32 recibos

Mismo camino que `python -m validator.run`, pero con el cache: normaliser -> classifier (replay) -> retriever + engine -> reporter. Qué mirar: 9 pass / 5 review / 18 flag, idéntico a `output/report.jsonl`.

In [29]:
reports = [{**reporter.report(r, RESULTS[r["receipt_id"]], C), "provider": C.meta["provider"]} for r in RECEIPTS]
assert reports == [committed[r["receipt_id"]] for r in RECEIPTS], "difiere de output/report.jsonl"
pd.Series(Counter(x["disposition"] for x in reports), name="recibos").reindex(["pass", "review", "flag"])

pass       9
review     5
flag      18
Name: recibos, dtype: int64

In [30]:
pd.DataFrame([{"receipt": x["receipt_id"], "merchant": BY_ID[x["receipt_id"]]["merchant_name"],
               "type": BY_ID[x["receipt_id"]]["submitted_expense_type"], "entity": BY_ID[x["receipt_id"]]["entity"],
               "disposition": x["disposition"],
               "exceptions": ", ".join(f"{e['rule_id']} ({e['severity']})" for e in x["exceptions"]),
               "signals": ", ".join(s["rule_id"] for s in x["review_signals"]),
               "superseded": ", ".join(s["rule_id"] for s in x["superseded"]),
               "considered": len(x["rules_considered"])} for x in reports])

,receipt,merchant,type,entity,disposition,exceptions,signals,superseded,considered
0,RCP-001,Harvest Table Kitchen,Meals - Individual,GLOBAL,pass,,,,10
1,RCP-002,Verona Grill,Client Entertainment,GLOBAL,pass,,,,12
2,RCP-003,Cornerstone Office Supply,Office Supplies,GLOBAL,pass,,,,6
3,RCP-004,Dockside Tavern & Grill,Meals - Team,GLOBAL,pass,,,,9
4,RCP-005,Riverside Inn,Lodging,GLOBAL,pass,,,,7
5,RCP-006,Cornerstone Office Supply,Computer Equipment,GLOBAL,pass,,,,6
6,RCP-007,Bluebird Cafe,Meals - Team,GLOBAL,pass,,,,8
7,RCP-008,Le Petit Marche,Meals - Team,EMEA,pass,,,MEAL-01,12
8,RCP-009,Crisp Cleaners,Travel - Other,GLOBAL,pass,,,,8
9,RCP-010,The Anchor House,Meals - Individual,GLOBAL,flag,ALC-01 (high),,,13


## 8. Evaluación — **aquí sí se leen las respuestas, solo para medir**

Desde aquí se importa `validator.gold` / `validator.evaluation.*`, que leen `data_eval/gold.jsonl`. Tres sistemas sobre el mismo engine:

- **A LLM:** el pipeline real (clasificador desde cache).
- **B ORACLE:** el engine con las categorías correctas: mide el engine solo.
- **C KEYWORD:** baseline de palabras clave, congelado antes de calificar (hash fijado en tests): lo que este proyecto reemplaza.

Se reusan las funciones de `validator.evaluation.evaluate`; al final se regenera `EVAL.md` en memoria y se compara byte a byte con el archivo committeado.

Ojo al leer: n=28. Los intervalos son Clopper-Pearson (exactos) y bootstrap por recibo. El engine se escribió viendo este dataset, así que B es consistencia, no generalización; la única medición ciega es el clasificador, y la mejor estimación de él es el held-out.

In [31]:
from validator.evaluation import evaluate as ev
from validator.evaluation import oracle

results, cats, extra, gold, _ = ev.run_systems()
ms = {s: ev.measures(results[s], cats[s], gold) for s in ev.SYSTEMS}
rows = [("disposición", "disposition"), ("excepciones precision", "exc_precision"), ("excepciones recall", "exc_recall"),
        ("evidencia exacta (líneas)", "evidence_exact"), ("señales recall", "sig_recall"),
        ("falsos positivos (de 9 pass)", "fp_of_pass"), ("violaciones perdidas (de 16 flag)", "missed_of_flag"),
        ("categoría por línea", "cat_acc"), ("restricted por línea", "restricted_acc")]
pd.DataFrame({s: {name: ev.fmt(ms[s][k]) if k in ms[s] else "gold" for name, k in rows} for s in ev.SYSTEMS})

,A LLM,B ORACLE,C KEYWORD
disposición,"28/28 = 100.0% [87.7%, 100.0%]","27/28 = 96.4% [81.7%, 99.9%]","21/28 = 75.0% [55.1%, 89.3%]"
excepciones precision,"26/26 = 100.0% [86.8%, 100.0%]","25/25 = 100.0% [86.3%, 100.0%]","19/21 = 90.5% [69.6%, 98.8%]"
excepciones recall,"26/26 = 100.0% [86.8%, 100.0%]","25/26 = 96.2% [80.4%, 99.9%]","19/26 = 73.1% [52.2%, 88.4%]"
evidencia exacta (líneas),"26/26 = 100.0% [86.8%, 100.0%]","25/25 = 100.0% [86.3%, 100.0%]","19/19 = 100.0% [82.4%, 100.0%]"
señales recall,"7/7 = 100.0% [59.0%, 100.0%]","7/7 = 100.0% [59.0%, 100.0%]","7/7 = 100.0% [59.0%, 100.0%]"
falsos positivos (de 9 pass),"0/9 = 0.0% [0.0%, 33.6%]","0/9 = 0.0% [0.0%, 33.6%]","1/9 = 11.1% [0.3%, 48.2%]"
violaciones perdidas (de 16 flag),"0/16 = 0.0% [0.0%, 20.6%]","1/16 = 6.2% [0.2%, 30.2%]","4/16 = 25.0% [7.3%, 52.4%]"
categoría por línea,"65/65 = 100.0% [94.5%, 100.0%]",gold,"49/65 = 75.4% [63.1%, 85.2%]"
restricted por línea,"65/65 = 100.0% [94.5%, 100.0%]",gold,"58/65 = 89.2% [79.1%, 95.6%]"


In [32]:
for s in ev.SYSTEMS:
    cm = ms[s]["confusion"]
    print(s)
    print(pd.DataFrame([[cm[(w, g)] for g in ev.DISP] for w in ev.DISP], index=[f"gold {w}" for w in ev.DISP], columns=ev.DISP), "\n")

A LLM
             flag  review  pass
gold flag      16       0     0
gold review     0       3     0
gold pass       0       0     9 

B ORACLE
             flag  review  pass
gold flag      15       0     1
gold review     0       3     0
gold pass       0       0     9 

C KEYWORD
             flag  review  pass
gold flag      12       1     3
gold review     1       1     1
gold pass       1       0     8 



**Oracle (B):** el engine con categorías perfectas falla un recibo, RCP-022. No es un bug de categoría: el destinatario (un funcionario público) solo está en el texto de la línea y las etiquetas gold no lo traen. El LLM sí lo extrae (`gift_for`), por eso A acierta.

In [33]:
mismatch, line = oracle.summary(results["B ORACLE"], gold)
print(line)
pd.DataFrame(mismatch, columns=["receipt", "what", "expected", "got"])

disposition 27/28 | exceptions matched 25/26, extra 0 | evidence exact 25/25 | signals missed 0, extra 0 | errors 0


,receipt,what,expected,got
0,RCP-022,disposition,flag,pass
1,RCP-022,missed exception,GIFT-04,-


**Keyword vs LLM**, línea por línea (pareado): dónde acierta uno y el otro no.

In [34]:
a, k = {x[5]: x for x in ms["A LLM"]["lines"]}, {x[5]: x for x in ms["C KEYWORD"]["lines"]}
desc = {li["line_id"]: li["description_raw"] for r in RECEIPTS for li in r["line_items"]}
diff = [{"line": i, "description": desc[i], "gold": a[i][1], "LLM": a[i][2], "keyword": k[i][2]}
        for i in sorted(a) if (a[i][1] == a[i][2]) != (k[i][1] == k[i][2])]
print("A correcta y C no en", ev.disposition_discordance(ms)[0], "recibos; al revés en", ev.disposition_discordance(ms)[1])
pd.DataFrame(diff)

A correcta y C no en 7 recibos; al revés en 0


,line,description,gold,LLM,keyword
0,RCP-002-L2,Chianti Classico bottle,alcohol,alcohol,other
1,RCP-003-L2,Legal pads 3-pack,office_supplies,office_supplies,other
2,RCP-004-L1,Beer battered cod & chips,food_beverage,food_beverage,alcohol
3,RCP-004-L2,Arnold Palmer,food_beverage,food_beverage,other
4,RCP-004-L3,Mojito sin alcohol,food_beverage,food_beverage,other
5,RCP-004-L4,Rum cake slice,food_beverage,food_beverage,alcohol
6,RCP-007-L2,Assorted soft drinks,food_beverage,food_beverage,other
7,RCP-008-L3,Cafe expresso,food_beverage,food_beverage,other
8,RCP-010-L2,Draft IPA 16oz,alcohol,alcohol,other
9,RCP-015-L2,Barolo 2016 bottle,alcohol,alcohol,other


**Intervalos.** 28/28 suena perfecto, pero con n=28 el límite inferior al 95% es ~88%. Por eso EVAL.md reporta siempre el intervalo.

In [35]:
for k_, n_ in ((28, 28), (27, 28), (21, 28), (0, 9)):
    lo, hi = ev.clopper_pearson(k_, n_)
    wl, wh = ev.wilson(k_, n_)
    print(f"{k_:>2}/{n_}: Clopper-Pearson [{lo:.3f}, {hi:.3f}]  Wilson [{wl:.3f}, {wh:.3f}]")
print("bootstrap disposición C:", ms["C KEYWORD"]["disposition_boot"])

28/28: Clopper-Pearson [0.877, 1.000]  Wilson [0.879, 1.000]
27/28: Clopper-Pearson [0.817, 0.999]  Wilson [0.823, 0.994]
21/28: Clopper-Pearson [0.551, 0.893]  Wilson [0.566, 0.873]
 0/9: Clopper-Pearson [0.000, 0.336]  Wilson [0.000, 0.299]
bootstrap disposición C: (0.5714285714285714, 0.8928571428571429)


**Held-out (estrés del clasificador).** 41 recibos / 144 líneas hechos a mano, con ítems que no están ni en el dataset ni en los ejemplos de la taxonomía. Las etiquetas son de un solo anotador. Primero verifico que el cache `.cache/heldout/` reproduce el registro committeado; luego las métricas.

In [36]:
from validator.evaluation import heldout

ho_receipts, expected = heldout.load()
ho = classifier.classify(ho_receipts, client=_NoNetwork(), cache_dir=heldout.CACHE)
assert not ho.meta["failures"], ho.meta["failures"]
records = ev.load_records()
rec = records["v2"]
same = all(ho.categories[x["line_id"]] == x["predicted"] and ho.meta["confidence"][x["line_id"]] == x["confidence"] for x in rec["lines"])
print(f"held-out replay: {len(ho_receipts)} recibos, {len(rec['lines'])} líneas, coincide con run_record.json: {same}")
pd.Series({name: ev.fmt(p) for name, p in ev.heldout_measures(rec).items()}, name="v2")

held-out replay: 41 recibos, 144 líneas, coincide con run_record.json: True


Exact category, all lines                            139/144 = 96.5% [92.1%, 98.9%]
Exact or listed alternative, all lines             144/144 = 100.0% [97.5%, 100.0%]
Exact, non-ambiguous lines                           130/133 = 97.7% [93.5%, 99.5%]
Ambiguous lines: acceptable answer or rated low      11/11 = 100.0% [71.5%, 100.0%]
Non-exact lines rated low (would raise EXC-03)             0/5 = 0.0% [0.0%, 52.2%]
Exact when confidence = high                       111/111 = 100.0% [96.7%, 100.0%]
Exact when confidence = medium                         20/25 = 80.0% [59.3%, 93.2%]
Exact when confidence = low                            5/5 = 100.0% [47.8%, 100.0%]
Stated recipients extracted with the right kind        5/5 = 100.0% [47.8%, 100.0%]
Lines with no stated recipient given one anyway           2/139 = 1.4% [0.2%, 5.1%]
Name: v2, dtype: str

In [37]:
regen = ev.report(ms, results, extra, gold, RECEIPTS, records, ho_receipts)
print("EVAL.md regenerado == output/EVAL.md:", regen == ev.OUT.read_text(encoding="utf-8"))

EVAL.md regenerado == output/EVAL.md: True


## 9. Dónde puede fallar

Dos casos concretos (también usan las respuestas, solo para medir).

**Caso 1, RCP-022 bajo el oracle.** Categorías perfectas y aun así `pass` en vez de `flag` (GIFT-04, critical): el engine necesita saber *a quién* se dio el regalo, y eso vive solo en texto libre. El pipeline lo acierta porque el LLM extrae el destinatario; o sea, una excepción crítica anticorrupción descansa en una lectura de texto libre de un LLM.

In [38]:
print("gold:", gold["RCP-022"]["labels"]["disposition"], [e["rule_id"] for e in gold["RCP-022"]["labels"]["expected_exceptions"]])
print("B ORACLE:", results["B ORACLE"]["RCP-022"].disposition, fired(results["B ORACLE"]["RCP-022"]))
print("A LLM:   ", results["A LLM"]["RCP-022"].disposition, fired(results["A LLM"]["RCP-022"]))
print("gift_for del LLM:", {k: v for k, v in C.gift_for.items() if k.startswith("RCP-022")})

gold: flag ['GIFT-04']
B ORACLE: pass []
A LLM:    flag [('GIFT-04', 'critical')]
gift_for del LLM: {'RCP-022-L1': ('government_official', 'R. Delgado'), 'RCP-022-L2': ('government_official', 'R. Delgado')}


**Caso 2, held-out: errores con confianza `medium`.** 5 de 144 líneas difieren del anotador, todas con `medium`. Como EXC-03 solo mira `low`, ninguna llega a un humano por la vía de incertidumbre; dos cruzan la frontera `restricted`. Además el LLM copió un destinatario a 2 líneas que no lo dicen.

In [39]:
f = ev.heldout_failures(rec, ho_receipts)
print("cruzan restricted:", [x["line_id"] for x in f["flips"]])
print("cambian el resultado del engine:", f["by_category"])
pd.DataFrame([{k: x[k] for k in ("line_id", "description_raw", "expected", "predicted", "confidence", "acceptable")} for x in f["wrong"]])

cruzan restricted: ['HO-008-L4', 'HO-011-L5']
cambian el resultado del engine: {'HO-008': 'PERS-01 cites HO-008-L1, HO-008-L2 instead of HO-008-L1, HO-008-L2, HO-008-L4', 'HO-011': 'review -> flag, LAUNDRY-01 no longer fires, PERS-01 fires on HO-011-L5', 'HO-017': 'PERS-02 no longer fires'}


,line_id,description_raw,expected,predicted,confidence,acceptable
0,HO-008-L4,Gel hydroalcoolique 100ml,personal_care,health_otc,medium,True
1,HO-011-L5,Shoe shine,laundry_service,personal_care,medium,True
2,HO-017-L2,"Embroidered uniform patch, company logo",clothing,gift,medium,True
3,HO-021-L2,Gravur,service_fee,other,medium,True
4,HO-026-L4,ウコンの力,food_beverage,health_otc,medium,True


In [40]:
pd.DataFrame([{"line_id": x["line_id"], "description_raw": x["description_raw"], "recipient_got": x["gift_got"]} for x in f["spurious"]])

,line_id,description_raw,recipient_got
0,HO-004-L1,Wine,"[client, J. Okafor]"
1,HO-018-L2,Emballage cadeau,"[client, Mme Lefèvre]"


**Límites conocidos** declarados en el código (`Limit:`):

In [41]:
for p in sorted((ROOT / "src" / "validator").rglob("*.py")):
    for i, text in enumerate(p.read_text(encoding="utf-8").splitlines(), 1):
        if "Limit:" in text:
            print(f"{p.relative_to(ROOT)}:{i}: {text.strip().lstrip('# ')}")

src/validator/classifier.py:250: Limit: re-fetched on each attempt while it fails (free, no key needed); cache a failure if it matters
src/validator/reporter.py:28: `**ID — title.** body` up to the first blank line. Limit: first paragraph only; MEAL-01's
src/validator/rules/gifts.py:12: Limit: who a line is for (client, employee, government official, and the name) lives only in free
src/validator/rules/gifts.py:19: Limit: a gift card for an employee is reported under GIFT-02 alone, not GIFT-03 as well.
src/validator/rules/gifts.py:20: Limit: on a "Client Gifts" receipt every line outside NOT_GIFTS is a gift line, so a tax line (if
src/validator/rules/health.py:6: Limit: prescription medication (the policy's medium branch) has no category: the taxonomy files it
src/validator/rules/meals.py:14: Limit: refunds and negative amounts get no special handling; a negative line lowers the day's total.
src/validator/rules/meals.py:27: Limit: only countries in the data; an unknown country raises K

### Qué debería picar un revisor

1. **EXC-03 solo con `low`.** Los errores del held-out vienen en `medium`. ¿Subir el umbral, o pedir una segunda categoría y mandar a revisión cuando las dos no coinciden en `restricted`? (sección 9, caso 2)
2. **EXC-03 baja de flag a review.** Una línea incierta sale de todas las reglas: una cerveza APAC incierta ya no dispara REG-01 (sección 5). ¿Debería el recibo heredar la peor severidad posible?
3. **Destinatario en texto libre.** GIFT-04 (critical) depende de que el LLM lea bien "City Procurement Office", y en held-out inventó destinatarios en 2 líneas. Se necesita un campo estructurado o medirlo como salida propia.
4. **Engine visto con las respuestas.** B ORACLE 27/28 es consistencia, no generalización. Los tests por regla y el held-out son la evidencia real; pedir receipts nuevos para el engine.
5. **n = 28, una corrida, dataset enriquecido (57% flag).** Mira los límites inferiores y la tabla de base rate en `output/EVAL.md`, no el 100%.
6. **Confianza 0.3/0.7/0.9 no calibrada.** Si alguien la usa como probabilidad, mal. Calibrar requiere más datos etiquetados.
7. **Cache como fuente de verdad.** Un cambio de prompt invalida todo el cache (bien), pero la reproducibilidad de los números depende de tener estos archivos; una nueva corrida del modelo puede dar otras respuestas.
8. **Retriever generoso.** La recall se garantiza con un `RuntimeError`, no con evidencia de que los predicados sean exactos; mira la tabla de `retriever.py` contra el texto de alcance de cada política.